# DFU V2 — Read-Only Whole-Drive Artifact Search
This notebook performs no training and does not delete, move, rename, or overwrite files in Google Drive. It searches mounted Drive roots for DFU trial artifacts, audits duplicate/conflicting candidates, reconstructs recoverable CSVs locally in `/content`, and downloads one ZIP.

In [ ]:
# ============================================================
# DFU V2 — READ-ONLY WHOLE-DRIVE ARTIFACT SEARCH + RECOVERY AUDIT
# NO TRAINING | NO DRIVE DELETE | NO DRIVE OVERWRITE
# Outputs are written only to /content and downloaded as a ZIP.
# ============================================================

from pathlib import Path
from google.colab import drive, files
import os
import json
import hashlib
import zipfile
import shutil
import time
import traceback
from collections import defaultdict

import pandas as pd

# -----------------------------
# 1) Mount Drive (read-only use)
# -----------------------------
drive.mount("/content/drive", force_remount=False)

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/content/drive/.shortcut-targets-by-id"),
]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]

if not SEARCH_ROOTS:
    raise RuntimeError("No mounted Google Drive roots were found.")

OUTPUT_DIR = Path("/content/DFU_READ_ONLY_ARTIFACT_AUDIT")
ZIP_PATH = Path("/content/DFU_READ_ONLY_ARTIFACT_AUDIT.zip")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAMES = {
    "COMPLETE.json",
    "test_predictions.csv",
    "TRIAL_VERIFICATION.json",
    "CHECKPOINT_RETENTION.json",
    "best_model_portable_fp16.pt",
    "last_resume.pt",
    "best_model.pt",
    "fold_seed_metrics.csv",
    "all_oof_predictions.csv",
    "RUN_PROGRESS.json",
    "FINAL_VERIFICATION.json",
    "RECONSTRUCTION_VALIDATION.json",
    "ARTIFACT_MANIFEST.json",
}

EXPECTED_MODELS = {
    "convnextv2_tiny",
    "mobilenetv3_large",
    "densenet121",
}
EXPECTED_SEEDS = {2026, 2027, 2028}
EXPECTED_FOLDS = {1, 2, 3, 4, 5}

EXPECTED_IDENTITIES = {
    (model, seed, fold)
    for fold in EXPECTED_FOLDS
    for seed in EXPECTED_SEEDS
    for model in EXPECTED_MODELS
}

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8")), ""
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

def normalize_identity(model, seed, fold):
    try:
        model = str(model)
        seed = int(seed)
        fold = int(fold)
        return model, seed, fold
    except Exception:
        return None

def identity_from_complete(path: Path):
    payload, error = safe_json(path)
    if payload is None:
        return None, None, error
    identity = normalize_identity(
        payload.get("model_key"),
        payload.get("seed"),
        payload.get("outer_fold"),
    )
    if identity is None:
        # Some code stores zero-based "fold".
        raw_fold = payload.get("fold")
        try:
            raw_fold = int(raw_fold)
            if raw_fold in {0, 1, 2, 3, 4}:
                raw_fold += 1
        except Exception:
            pass
        identity = normalize_identity(
            payload.get("model_key"),
            payload.get("seed"),
            raw_fold,
        )
    return identity, payload, error

def identity_from_predictions(path: Path):
    required = {"model_key", "seed", "outer_fold"}
    try:
        frame = pd.read_csv(
            path,
            usecols=lambda c: c in required,
            low_memory=False,
        )
        if not required.issubset(frame.columns):
            return None, None, f"Missing identity columns: {sorted(required - set(frame.columns))}"
        unique = frame[list(required)].drop_duplicates()
        if len(unique) != 1:
            return None, len(frame), f"Expected one identity, found {len(unique)}"
        row = unique.iloc[0]
        identity = normalize_identity(
            row["model_key"],
            row["seed"],
            row["outer_fold"],
        )
        return identity, len(frame), ""
    except Exception as exc:
        return None, None, f"{type(exc).__name__}: {exc}"

# ---------------------------------------
# 2) Targeted read-only recursive search
# ---------------------------------------
started = time.time()
inventory = []
scan_errors = []
dirs_scanned = 0
files_seen = 0

print("Search roots:")
for root in SEARCH_ROOTS:
    print(" -", root)

for root in SEARCH_ROOTS:
    for current, dirs, names in os.walk(root, topdown=True, followlinks=False):
        dirs_scanned += 1

        # Skip obvious non-result system/cache directories.
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".Trash",
                ".cache",
                "__pycache__",
                ".ipynb_checkpoints",
                "node_modules",
            }
        ]

        for name in names:
            files_seen += 1
            if name not in TARGET_NAMES and not name.endswith((".tmp", ".partial")):
                continue

            path = Path(current) / name
            try:
                stat = path.stat()
                row = {
                    "search_root": str(root),
                    "path": str(path),
                    "name": name,
                    "parent": str(path.parent),
                    "bytes": int(stat.st_size),
                    "modified_time": float(stat.st_mtime),
                    "modified_iso": time.strftime(
                        "%Y-%m-%dT%H:%M:%S",
                        time.localtime(stat.st_mtime),
                    ),
                    "is_temporary": name.endswith((".tmp", ".partial")),
                    "read_error": "",
                }
            except Exception as exc:
                row = {
                    "search_root": str(root),
                    "path": str(path),
                    "name": name,
                    "parent": str(path.parent),
                    "bytes": None,
                    "modified_time": None,
                    "modified_iso": None,
                    "is_temporary": name.endswith((".tmp", ".partial")),
                    "read_error": f"{type(exc).__name__}: {exc}",
                }
            inventory.append(row)

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(OUTPUT_DIR / "artifact_inventory.csv", index=False)

# ------------------------------------------------
# 3) Build per-trial candidate records from parents
# ------------------------------------------------
candidate_parents = sorted({
    Path(row["parent"])
    for row in inventory
    if row["name"] in {
        "COMPLETE.json",
        "test_predictions.csv",
        "TRIAL_VERIFICATION.json",
        "CHECKPOINT_RETENTION.json",
        "best_model_portable_fp16.pt",
        "last_resume.pt",
        "best_model.pt",
    }
})

trial_rows = []

for parent in candidate_parents:
    complete = parent / "COMPLETE.json"
    predictions = parent / "test_predictions.csv"
    portable = parent / "best_model_portable_fp16.pt"
    verification = parent / "TRIAL_VERIFICATION.json"
    retention = parent / "CHECKPOINT_RETENTION.json"
    resume = parent / "last_resume.pt"
    best_full = parent / "best_model.pt"

    complete_identity = None
    complete_payload = None
    complete_error = ""
    if complete.is_file():
        complete_identity, complete_payload, complete_error = identity_from_complete(complete)

    prediction_identity = None
    prediction_rows = None
    prediction_error = ""
    if predictions.is_file():
        prediction_identity, prediction_rows, prediction_error = identity_from_predictions(predictions)

    identity = complete_identity or prediction_identity
    identity_conflict = (
        complete_identity is not None
        and prediction_identity is not None
        and complete_identity != prediction_identity
    )

    model_key = identity[0] if identity else None
    seed = identity[1] if identity else None
    outer_fold = identity[2] if identity else None

    score = (
        4 * int(complete.is_file())
        + 4 * int(predictions.is_file())
        + 2 * int(portable.is_file())
        + int(verification.is_file())
        + int(retention.is_file())
    )

    row = {
        "trial_dir": str(parent),
        "model_key": model_key,
        "seed": seed,
        "outer_fold": outer_fold,
        "identity_conflict": identity_conflict,
        "complete_exists": complete.is_file(),
        "predictions_exists": predictions.is_file(),
        "portable_exists": portable.is_file(),
        "verification_exists": verification.is_file(),
        "retention_exists": retention.is_file(),
        "resume_exists": resume.is_file(),
        "best_full_exists": best_full.is_file(),
        "prediction_rows": prediction_rows,
        "candidate_score": score,
        "complete_error": complete_error,
        "prediction_error": prediction_error,
        "complete_path": str(complete) if complete.is_file() else "",
        "predictions_path": str(predictions) if predictions.is_file() else "",
        "portable_path": str(portable) if portable.is_file() else "",
        "verification_path": str(verification) if verification.is_file() else "",
        "modified_time": max(
            [
                p.stat().st_mtime
                for p in [complete, predictions, portable, verification, retention, resume, best_full]
                if p.is_file()
            ]
            or [0]
        ),
    }
    trial_rows.append(row)

trials_df = pd.DataFrame(trial_rows)
if len(trials_df):
    trials_df = trials_df.sort_values(
        ["candidate_score", "modified_time"],
        ascending=[False, False],
    )
trials_df.to_csv(OUTPUT_DIR / "trial_candidates.csv", index=False)

# -------------------------------------------------
# 4) Select the best valid candidate per identity
# -------------------------------------------------
valid_candidates = trials_df.copy() if len(trials_df) else pd.DataFrame()

if len(valid_candidates):
    valid_candidates = valid_candidates[
        valid_candidates["model_key"].notna()
        & valid_candidates["seed"].notna()
        & valid_candidates["outer_fold"].notna()
        & valid_candidates["complete_exists"]
        & valid_candidates["predictions_exists"]
        & ~valid_candidates["identity_conflict"]
        & (valid_candidates["complete_error"] == "")
        & (valid_candidates["prediction_error"] == "")
    ].copy()

selected_rows = []
duplicate_rows = []
conflicts = []

if len(valid_candidates):
    for identity_values, group in valid_candidates.groupby(
        ["model_key", "seed", "outer_fold"],
        dropna=False,
    ):
        group = group.sort_values(
            ["candidate_score", "modified_time"],
            ascending=[False, False],
        ).copy()

        # Compute hashes only for valid small scientific artifacts.
        hashes = []
        for _, candidate in group.iterrows():
            try:
                complete_sha = sha256_file(Path(candidate["complete_path"]))
                prediction_sha = sha256_file(Path(candidate["predictions_path"]))
                hashes.append((complete_sha, prediction_sha))
            except Exception as exc:
                hashes.append((f"ERROR:{exc}", f"ERROR:{exc}"))

        group["complete_sha256"] = [h[0] for h in hashes]
        group["predictions_sha256"] = [h[1] for h in hashes]

        winner = group.iloc[0].to_dict()
        winner["candidate_count_for_identity"] = int(len(group))
        winner["duplicate_content_identical"] = (
            len(set(hashes)) == 1 if len(hashes) > 1 else True
        )
        selected_rows.append(winner)

        if len(group) > 1:
            duplicate_rows.extend(group.to_dict("records"))
            if len(set(hashes)) > 1:
                conflicts.append({
                    "model_key": identity_values[0],
                    "seed": int(identity_values[1]),
                    "outer_fold": int(identity_values[2]),
                    "candidate_count": int(len(group)),
                    "reason": "Multiple valid candidates have different hashes.",
                    "paths": group["trial_dir"].tolist(),
                })

selected_df = pd.DataFrame(selected_rows)
if len(selected_df):
    selected_df = selected_df.sort_values(
        ["outer_fold", "seed", "model_key"]
    ).reset_index(drop=True)

selected_df.to_csv(OUTPUT_DIR / "selected_trial_artifacts.csv", index=False)
pd.DataFrame(duplicate_rows).to_csv(
    OUTPUT_DIR / "duplicate_identity_candidates.csv",
    index=False,
)
pd.DataFrame(conflicts).to_csv(
    OUTPUT_DIR / "conflicting_identity_candidates.csv",
    index=False,
)

observed_identities = {
    (str(r.model_key), int(r.seed), int(r.outer_fold))
    for r in selected_df.itertuples(index=False)
} if len(selected_df) else set()

missing_identities = sorted(
    EXPECTED_IDENTITIES - observed_identities,
    key=lambda x: (x[2], x[1], x[0]),
)
unexpected_identities = sorted(
    observed_identities - EXPECTED_IDENTITIES,
    key=lambda x: (x[2], x[1], x[0]),
)

missing_df = pd.DataFrame([
    {"model_key": m, "seed": s, "outer_fold": f}
    for m, s, f in missing_identities
])
missing_df.to_csv(OUTPUT_DIR / "missing_trial_identities.csv", index=False)

unexpected_df = pd.DataFrame([
    {"model_key": m, "seed": s, "outer_fold": f}
    for m, s, f in unexpected_identities
])
unexpected_df.to_csv(OUTPUT_DIR / "unexpected_trial_identities.csv", index=False)

# ----------------------------------------------------------
# 5) Reconstruct CSVs locally from the selected candidates
# ----------------------------------------------------------
metric_rows = []
prediction_frames = []
reconstruction_errors = []

for row in selected_rows:
    identity = (
        str(row["model_key"]),
        int(row["seed"]),
        int(row["outer_fold"]),
    )
    try:
        complete_path = Path(row["complete_path"])
        predictions_path = Path(row["predictions_path"])

        payload, error = safe_json(complete_path)
        if payload is None:
            raise ValueError(error)

        pred = pd.read_csv(predictions_path, low_memory=False)
        required_columns = {
            "model_key",
            "seed",
            "outer_fold",
            "image_id",
            "label",
            "prob_calibrated",
            "pred",
        }
        missing_columns = required_columns - set(pred.columns)
        if missing_columns:
            raise ValueError(
                f"Missing prediction columns: {sorted(missing_columns)}"
            )

        pred_identities = {
            normalize_identity(m, s, f)
            for m, s, f in pred[
                ["model_key", "seed", "outer_fold"]
            ].drop_duplicates().itertuples(index=False, name=None)
        }
        if pred_identities != {identity}:
            raise ValueError(
                f"Prediction identity mismatch: {pred_identities} != {identity}"
            )

        payload["artifact_trial_dir"] = row["trial_dir"]
        payload["complete_sha256"] = row.get("complete_sha256", "")
        payload["predictions_sha256"] = row.get("predictions_sha256", "")
        metric_rows.append(payload)
        prediction_frames.append(pred)

    except Exception as exc:
        reconstruction_errors.append({
            "model_key": identity[0],
            "seed": identity[1],
            "outer_fold": identity[2],
            "trial_dir": row["trial_dir"],
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc()[-2000:],
        })

metrics_df = pd.DataFrame(metric_rows)
predictions_df = (
    pd.concat(prediction_frames, ignore_index=True)
    if prediction_frames
    else pd.DataFrame()
)

if len(metrics_df):
    metrics_df = metrics_df.sort_values(
        ["outer_fold", "seed", "model_key"]
    ).reset_index(drop=True)
    metrics_df.to_csv(
        OUTPUT_DIR / "fold_seed_metrics_RECOVERED.csv",
        index=False,
    )

if len(predictions_df):
    predictions_df = predictions_df.sort_values(
        ["outer_fold", "seed", "model_key", "image_id"]
    ).reset_index(drop=True)
    predictions_df.to_csv(
        OUTPUT_DIR / "all_oof_predictions_RECOVERED.csv",
        index=False,
    )

pd.DataFrame(reconstruction_errors).to_csv(
    OUTPUT_DIR / "reconstruction_errors.csv",
    index=False,
)

# ------------------------------------
# 6) Generate final read-only report
# ------------------------------------
temporary_files = (
    inventory_df[inventory_df["is_temporary"] == True]
    if len(inventory_df)
    else pd.DataFrame()
)
temporary_bytes = int(
    pd.to_numeric(
        temporary_files.get("bytes", pd.Series(dtype=float)),
        errors="coerce",
    ).fillna(0).sum()
) if len(temporary_files) else 0

status = (
    "PASS_45_RECOVERED"
    if (
        len(observed_identities) == 45
        and not missing_identities
        and not unexpected_identities
        and not conflicts
        and not reconstruction_errors
        and len(metrics_df) == 45
    )
    else "PARTIAL_OR_CONFLICTED"
)

report = {
    "mode": "READ_ONLY_DRIVE_SEARCH",
    "drive_modified": False,
    "search_roots": [str(p) for p in SEARCH_ROOTS],
    "search_seconds": round(time.time() - started, 2),
    "directories_scanned": dirs_scanned,
    "files_seen": files_seen,
    "target_artifacts_found": int(len(inventory_df)),
    "trial_candidate_directories": int(len(trials_df)),
    "valid_unique_trial_identities": int(len(observed_identities)),
    "expected_trial_identities": 45,
    "missing_trial_count": int(len(missing_identities)),
    "unexpected_trial_count": int(len(unexpected_identities)),
    "conflicting_identity_count": int(len(conflicts)),
    "reconstruction_error_count": int(len(reconstruction_errors)),
    "recovered_metric_rows": int(len(metrics_df)),
    "recovered_prediction_rows": int(len(predictions_df)),
    "temporary_file_count": int(len(temporary_files)),
    "temporary_file_bytes": temporary_bytes,
    "status": status,
    "output_location": str(OUTPUT_DIR),
}

(OUTPUT_DIR / "READ_ONLY_SEARCH_REPORT.json").write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8",
)

print("\n" + "=" * 72)
print("DFU READ-ONLY WHOLE-DRIVE SEARCH RESULT")
print("=" * 72)
print(json.dumps(report, indent=2))
print("\nDrive files were not deleted, moved, renamed, or overwritten.")

if missing_identities:
    print("\nStill missing:")
    for model, seed, fold in missing_identities:
        print(f"  Fold {fold} | seed {seed} | {model}")

# ------------------------
# 7) ZIP local audit files
# ------------------------
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUTPUT_DIR))

print("\nAudit ZIP:", ZIP_PATH)
files.download(str(ZIP_PATH))
